### 4.2 Modeling/4.3 Feature selection

In [45]:
# Customized the train_test_split for augmentated dataset

In [46]:
import numpy as np
import pandas as pd
from sklearn.utils import shuffle

def balanced_group_train_test_split(X, y, group_col='augmented_from_uniprot_id', 
                                   test_size=0.2, random_state=None):
    """
    Splits data into balanced train/test sets while keeping augmented groups together.
    
    Parameters:
    X (pd.DataFrame): Feature dataframe containing the group column
    y (pd.Series): Target labels
    group_col (str): Name of the column containing group IDs
    test_size (float): Proportion of data for testing (default 0.2)
    random_state (int): Random seed for reproducibility
    
    Returns:
    (pd.DataFrame, pd.DataFrame, pd.Series, pd.Series): X_train, X_test, y_train, y_test
    """
    # Validate inputs
    if group_col not in X.columns:
        raise ValueError(f"Group column '{group_col}' not found in feature dataframe")
        
    if len(X) != len(y):
        raise ValueError("X and y must have the same number of samples")

    rng = np.random.RandomState(random_state)
    combined = X.copy()
    combined['_label'] = y.values

    # Create group metadata
    group_meta = combined.groupby(group_col).agg({
        '_label': 'first',
        group_col: 'count'
    }).rename(columns={group_col: 'count'}).reset_index()

    train_groups, test_groups = [], []

    # Stratified group splitting
    for label in group_meta['_label'].unique():
        label_groups = group_meta[group_meta['_label'] == label]
        total_samples = label_groups['count'].sum()
        test_samples_target = int(total_samples * test_size)
        
        label_groups = shuffle(label_groups, random_state=rng)
        
        # Select groups for test set
        cumulative, selected = 0, []
        for _, row in label_groups.iterrows():
            if cumulative >= test_samples_target:
                break
            selected.append(row[group_col])
            cumulative += row['count']
        
        # Assign remaining groups to train
        remaining = list(set(label_groups[group_col]) - set(selected))
        test_groups.extend(selected)
        train_groups.extend(remaining)

    # Create splits
    train_mask = combined[group_col].isin(train_groups)
    test_mask = combined[group_col].isin(test_groups)
    
    # Return splits with group column removed from features
    return (
            X[train_mask].drop(columns=[group_col]),
            X[test_mask].drop(columns=[group_col]),
            y[train_mask],
            y[test_mask])

    # return (
    #         X[train_mask],
    #         X[test_mask],
    #         y[train_mask],
    #         y[test_mask])

In [47]:
df = pd.read_pickle("final_dataset_with_all_new.pkl")

y = df["label"]
target = "label"
features = list(df.columns)

features.remove(target)

# Select all columns starting with 'onehot' for features
x = df[features]

# Split data into train and test sets
x_train, x_test, y_train, y_test = balanced_group_train_test_split(
    x,
    y,
    group_col='augmentated from uniprot ID',
    test_size=0.2,
    random_state=42
)

In [48]:
# training_uniprot = set(x_train["augmentated from uniprot ID"])
# testing_uniprot = set(x_test["augmentated from uniprot ID"])
# # Find any overlapping UniProt IDs
# overlap = training_uniprot.intersection(testing_uniprot)

# # Check if there's any overlap
# if len(overlap) > 0:
#     print(f"Found {len(overlap)} overlapping UniProt IDs between training and test sets")
#     print("First few overlapping IDs:", list(overlap)[:5])
# else:
#     print("No overlap found between training and test sets - good to proceed!")

task 36: modeling direct from only PLM embedding, we first compare using embeddings from either ESM2 or PTM-mamba to represent the whole protein or the residue embeddings, 

In [49]:
import pandas as pd
import pickle
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

def make_pipeline(classifier):
    """Creates a machine learning pipeline with a Random Forest Classifier.

    Returns:
        Pipeline: A scikit-learn pipeline with a RandomForestClassifier as the final step.
    """
    steps = [("classifier", classifier)]
    pipe = Pipeline(steps)
    return pipe

def train(df, classifier):
    """Trains a Random Forest model using the specified DataFrame.

    Parameters:
        df (pd.DataFrame): The DataFrame containing the features and target variable.

    Returns:
        Pipeline: The trained RandomForest model pipeline.
    """
    # We are predicting the protein labels
    y = df["label"]
    target = "label"
    features = list(df.columns)
    
    features.remove(target)

    # Select all columns starting with 'onehot' for features
    x = df[features]

    # Split data into train and test sets
    x_train, x_test, y_train, y_test = balanced_group_train_test_split(
        x,
        y,
        group_col='augmentated from uniprot ID',
        test_size=0.2,
        random_state=42
    )

    # Count positive and negative labels in y_train and y_test
    y_train_counts = y_train.value_counts()
    y_test_counts = y_test.value_counts()

    print("Label counts in y_train:")
    print(y_train_counts)

    print("\nLabel counts in y_test:")
    print(y_test_counts)

    trained_model = make_pipeline(classifier)
    trained_model.fit(x_train, y_train)

    # Make predictions
    y_train_pred = trained_model.predict(x_train)
    y_test_pred = trained_model.predict(x_test)

    # Calculate model metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    train_report = classification_report(y_train, y_train_pred, zero_division=1)
    test_report = classification_report(y_test, y_test_pred, zero_division=1)

    # Print metrics in a readable format
    print(f"Model {classifier} Performance Metrics:\n")

    print("Training Data Metrics:")
    print(f"Accuracy: {train_accuracy:.2f}")
    print("Classification Report:")
    print(train_report)  # This prints the report as a table

    print("\nTest Data Metrics:")
    print(f"Accuracy: {test_accuracy:.2f}")
    print("Classification Report:")
    print(test_report)  # This prints the report as a table

    print("Successfully trained the model.")
    return trained_model

In [50]:
training_dataset_df = pd.read_pickle("final_dataset_with_all_new.pkl")
training_dataset_df = training_dataset_df[training_dataset_df["usage"] == "training"]

In [51]:
training_dataset_df.head(5)

,Unnamed: 0,uniprot_id,sequence,site,label,augmentated from uniprot ID,human Site and Mutation,usage,unique_id,deephase_phys_multi,...,ptmmamba_residue_embedding_759,ptmmamba_residue_embedding_760,ptmmamba_residue_embedding_761,ptmmamba_residue_embedding_762,ptmmamba_residue_embedding_763,ptmmamba_residue_embedding_764,ptmmamba_residue_embedding_765,ptmmamba_residue_embedding_766,ptmmamba_residue_embedding_767,phospho_score
0,0,H7C5C7,XVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMNKTGFVPSNYVER...,73,1,P16333,S73s,training,H7C5C7_73_XVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMNKT...,0.203,...,-0.930157,-0.784488,1.378070,1.125187,-0.962103,0.749272,-0.813084,0.488083,1.446624,0.526
1,1,A0A5F9CW23,MAEEVVVVAKFDYVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMN...,85,1,P16333,S85s,training,A0A5F9CW23_85_MAEEVVVVAKFDYVAQQEQELDIKKNERLWLL...,0.185,...,-0.833150,-0.543848,1.270835,1.192153,-0.817320,0.709415,-1.378276,0.626975,2.133350,0.579
2,2,Q5RF68,MDWLNVFKDFFSIGKVKRKPsVPDSASPADDSFVDPGERLYDLNMP...,21,1,P16333,S21s,training,Q5RF68_21_MDWLNVFKDFFSIGKVKRKPsVPDSASPADDSFVDP...,0.096,...,-1.474621,-0.104392,1.062815,0.442375,-0.308503,-0.050680,-1.171706,1.861005,0.189925,0.571
3,3,H3A391,MTEEVTVIAKFDYVAQQEQELDIKKNEKLLLLDDSKSWWRVRNSMN...,85,1,P16333,S85s,training,H3A391_85_MTEEVTVIAKFDYVAQQEQELDIKKNEKLLLLDDSK...,0.108,...,-0.708835,-0.368524,0.630111,0.287115,0.439652,0.259266,-1.870489,2.179887,0.830376,0.379
4,4,A0A8J1IQP6,MSLNGRRGSGRPGYYYRLIGRSQLQRQRSRSRSRNRPARRESPPER...,725,1,Q5VWQ8,S725s,training,A0A8J1IQP6_725_MSLNGRRGSGRPGYYYRLIGRSQLQRQRSRS...,0.282,...,-0.940106,-0.094523,0.474609,-0.267281,-0.266375,0.243640,-0.773728,1.224039,-0.136662,0.374


In [52]:
columns = training_dataset_df.columns
columns

Index(['Unnamed: 0', 'uniprot_id', 'sequence', 'site', 'label',
       'augmentated from uniprot ID', 'human Site and Mutation', 'usage',
       'unique_id', 'deephase_phys_multi',
       ...
       'ptmmamba_residue_embedding_759', 'ptmmamba_residue_embedding_760',
       'ptmmamba_residue_embedding_761', 'ptmmamba_residue_embedding_762',
       'ptmmamba_residue_embedding_763', 'ptmmamba_residue_embedding_764',
       'ptmmamba_residue_embedding_765', 'ptmmamba_residue_embedding_766',
       'ptmmamba_residue_embedding_767', 'phospho_score'],
      dtype='object', length=4454)

In [53]:
always_exclude_columns = ['Unnamed: 0', 'uniprot_id', 'sequence', 'site',  'human Site and Mutation', 'usage',
       'unique_id', "anchor_score"]

In [54]:
esm_training_dataset_df = training_dataset_df.filter(regex=r'^(esm_protein_embedding_|esm_residue_embedding_|label|augmentated from uniprot ID)')
ptmmamba_training_dataset_df = training_dataset_df.filter(regex=r'^(ptmmamba_residue_embedding|ptmmamba_protein_embedding_position|label|augmentated from uniprot ID)')

In [55]:
esm_training_dataset_df.head(5)

,label,augmentated from uniprot ID,esm_protein_embedding_0,esm_protein_embedding_1,esm_protein_embedding_2,esm_protein_embedding_3,esm_protein_embedding_4,esm_protein_embedding_5,esm_protein_embedding_6,esm_protein_embedding_7,...,esm_residue_embedding_1270,esm_residue_embedding_1271,esm_residue_embedding_1272,esm_residue_embedding_1273,esm_residue_embedding_1274,esm_residue_embedding_1275,esm_residue_embedding_1276,esm_residue_embedding_1277,esm_residue_embedding_1278,esm_residue_embedding_1279
0,1,P16333,0.021365,-0.020171,0.021621,0.095043,-0.057202,-0.050794,0.071949,0.006663,...,0.156998,-0.048794,-0.288164,-0.169802,-0.087454,-0.099428,-0.027398,-0.170118,-0.108479,-0.087466
1,1,P16333,0.024337,-0.044602,0.026486,0.129851,-0.089049,-0.042135,0.083374,0.002857,...,0.155553,-0.035488,-0.296511,-0.183268,-0.062212,-0.104337,-0.041655,-0.249533,-0.119220,-0.131040
2,1,P16333,0.019708,-0.037163,0.012426,0.091465,-0.085954,-0.071278,0.061197,0.026536,...,0.057347,-0.025978,-0.107604,-0.075288,-0.089051,-0.013340,0.077709,-0.006540,-0.176868,-0.109933
3,1,P16333,0.016691,-0.067765,-0.026445,0.096103,-0.135539,-0.088490,0.060594,-0.063513,...,0.172102,-0.006700,-0.067678,-0.127333,-0.010801,-0.093588,-0.160996,-0.111260,-0.075962,-0.120771
4,1,Q5VWQ8,0.028754,-0.000879,0.022422,0.037734,-0.101758,-0.097661,0.048510,0.022037,...,0.104533,0.161240,-0.105866,-0.053904,-0.058608,-0.270229,-0.001867,0.031717,-0.097747,-0.078485


In [56]:
ptmmamba_training_dataset_df.head(5)

,label,augmentated from uniprot ID,ptmmamba_protein_embedding_position_0,ptmmamba_protein_embedding_position_1,ptmmamba_protein_embedding_position_2,ptmmamba_protein_embedding_position_3,ptmmamba_protein_embedding_position_4,ptmmamba_protein_embedding_position_5,ptmmamba_protein_embedding_position_6,ptmmamba_protein_embedding_position_7,...,ptmmamba_residue_embedding_758,ptmmamba_residue_embedding_759,ptmmamba_residue_embedding_760,ptmmamba_residue_embedding_761,ptmmamba_residue_embedding_762,ptmmamba_residue_embedding_763,ptmmamba_residue_embedding_764,ptmmamba_residue_embedding_765,ptmmamba_residue_embedding_766,ptmmamba_residue_embedding_767
0,1,P16333,0.306219,0.075793,0.279163,0.129615,-0.124552,0.055463,-0.138853,-0.701024,...,0.905219,-0.930157,-0.784488,1.378070,1.125187,-0.962103,0.749272,-0.813084,0.488083,1.446624
1,1,P16333,0.164579,-0.161845,0.364336,0.196843,-0.142813,-0.032246,-0.274839,-0.841149,...,1.007017,-0.833150,-0.543848,1.270835,1.192153,-0.817320,0.709415,-1.378276,0.626975,2.133350
2,1,P16333,0.119370,-0.170645,0.069318,0.130563,0.066603,-0.061079,0.007130,-0.659120,...,0.972614,-1.474621,-0.104392,1.062815,0.442375,-0.308503,-0.050680,-1.171706,1.861005,0.189925
3,1,P16333,0.252884,-0.104365,0.170443,0.096454,0.100202,-0.116983,-0.210762,-0.806122,...,0.809762,-0.708835,-0.368524,0.630111,0.287115,0.439652,0.259266,-1.870489,2.179887,0.830376
4,1,Q5VWQ8,0.218763,0.069228,0.227428,0.684456,0.372327,-0.193287,-0.635183,-0.696890,...,0.046559,-0.940106,-0.094523,0.474609,-0.267281,-0.266375,0.243640,-0.773728,1.224039,-0.136662


In [57]:
plm_feature_df_lst = [esm_training_dataset_df, ptmmamba_training_dataset_df]
plm_name_lst = ["esm", "ptmmamba"]

In [58]:
for name, loaded_df in zip(plm_name_lst, plm_feature_df_lst):
    print(f"Modeling using {name}")
    model = train(loaded_df, RandomForestClassifier(max_depth=7))

Modeling using esm
Label counts in y_train:
label
0    966
1    841
Name: count, dtype: int64

Label counts in y_test:
label
0    233
1    226
Name: count, dtype: int64
Model RandomForestClassifier(max_depth=7) Performance Metrics:

Training Data Metrics:
Accuracy: 0.97
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       966
           1       0.96      0.97      0.96       841

    accuracy                           0.97      1807
   macro avg       0.97      0.97      0.97      1807
weighted avg       0.97      0.97      0.97      1807


Test Data Metrics:
Accuracy: 0.79
Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.93      0.82       233
           1       0.90      0.64      0.75       226

    accuracy                           0.79       459
   macro avg       0.81      0.78      0.78       459
weighted avg       0.81      0.79      0.78      

In [59]:
import gc
gc.collect()

1221

task 37: Then we tested modeling from multi-scale biology features plus onehot encoding for the -7 to +7 motif, the result show that multi-scale biology features also can be used to predict the 14-3-3 binding when compare with only use onehot encoded motif feature. 

In [60]:
# we first test to only use onehot encoding for the prediction

In [61]:
onehot_training_dataset_df = training_dataset_df.filter(regex=r'^(onehot_|label|augmentated from uniprot ID)')
onehot_training_dataset_df

,label,augmentated from uniprot ID,onehot_-7_A,onehot_-7_C,onehot_-7_D,onehot_-7_E,onehot_-7_F,onehot_-7_G,onehot_-7_H,onehot_-7_I,...,onehot_7_P,onehot_7_Q,onehot_7_R,onehot_7_S,onehot_7_T,onehot_7_V,onehot_7_W,onehot_7_Y,onehot_7_s,onehot_7_t
0,1,P16333,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,P16333,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,P16333,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,P16333,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,Q5VWQ8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2261,0,O15417,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2262,0,Q9BYW2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2263,0,Q8N3K9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2264,0,Q6ZRS2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [62]:
print(f"Modeling only use onehot encoding features")
model = train(onehot_training_dataset_df, RandomForestClassifier(max_depth=7))

Modeling only use onehot encoding features
Label counts in y_train:
label
0    966
1    841
Name: count, dtype: int64

Label counts in y_test:
label
0    233
1    226
Name: count, dtype: int64
Model RandomForestClassifier(max_depth=7) Performance Metrics:

Training Data Metrics:
Accuracy: 0.90
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.92      0.91       966
           1       0.90      0.89      0.90       841

    accuracy                           0.90      1807
   macro avg       0.90      0.90      0.90      1807
weighted avg       0.90      0.90      0.90      1807


Test Data Metrics:
Accuracy: 0.83
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       233
           1       0.85      0.80      0.82       226

    accuracy                           0.83       459
   macro avg       0.83      0.83      0.83       459
weighted avg       0.83  

In [63]:
# then we test use both onehot encode and biological features

In [64]:
excluded_prefixes = (
    'esm_protein_embedding_',
    'esm_residue_embedding_',
    'ptmmamba_residue_embedding_',
    'ptmmamba_protein_embedding_'
)

# Create a list of columns to keep (those that don't start with excluded prefixes)
columns_to_keep = [col for col in training_dataset_df.columns if not any(col.startswith(prefix) for prefix in excluded_prefixes)]
columns_to_keep = [col for col in columns_to_keep if col not in always_exclude_columns]
# Now select those columns from your DataFrame
biology_training_dataset_df = training_dataset_df[columns_to_keep]

In [65]:
biology_training_dataset_df

,label,augmentated from uniprot ID,deephase_phys_multi,deephase_w2v_multi,deephase_score,iupred_score,onehot_-7_A,onehot_-7_C,onehot_-7_D,onehot_-7_E,...,SCD,kappa,FCR,NCPR,fK,fR,fE,fD,faro,phospho_score
0,1,P16333,0.203,0.160,0.182,0.541878,0.0,0.0,0.0,0.0,...,-0.950,0.363,0.385,-0.077,0.077,0.077,0.000,0.154,0.000,0.526
1,1,P16333,0.185,0.173,0.179,0.509769,0.0,0.0,0.0,0.0,...,-0.546,-1.000,0.600,0.200,0.200,0.200,0.000,0.000,0.000,0.579
2,1,P16333,0.096,0.129,0.113,0.465241,0.0,0.0,0.0,0.0,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.571
3,1,P16333,0.108,0.352,0.230,0.566480,0.0,0.0,0.0,0.0,...,-0.946,0.250,0.381,-0.095,0.048,0.095,0.000,0.190,0.000,0.379
4,1,Q5VWQ8,0.282,0.742,0.512,0.837511,0.0,0.0,0.0,0.0,...,-0.158,0.382,0.231,-0.077,0.038,0.038,0.077,0.038,0.038,0.374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2261,0,O15417,0.329,0.774,0.552,0.798249,0.0,0.0,0.0,0.0,...,14.619,0.152,0.278,-0.160,0.024,0.036,0.154,0.059,0.018,0.080
2262,0,Q9BYW2,0.283,0.776,0.530,0.831250,0.0,0.0,0.0,0.0,...,22.077,0.190,0.389,-0.246,0.063,0.008,0.214,0.095,0.000,0.274
2263,0,Q8N3K9,0.203,0.726,0.464,0.566480,0.0,0.0,0.0,0.0,...,9.433,0.217,0.229,-0.178,0.034,0.000,0.178,0.025,0.059,0.063
2264,0,Q6ZRS2,0.411,0.762,0.587,0.930790,0.0,0.0,0.0,0.0,...,0.416,0.164,0.080,0.034,0.017,0.040,0.011,0.006,0.011,0.155


In [66]:
print(f"Modeling using multi-scale biology features")
model = train(biology_training_dataset_df, RandomForestClassifier(max_depth=7))

Modeling using multi-scale biology features
Label counts in y_train:
label
0    966
1    841
Name: count, dtype: int64

Label counts in y_test:
label
0    233
1    226
Name: count, dtype: int64
Model RandomForestClassifier(max_depth=7) Performance Metrics:

Training Data Metrics:
Accuracy: 0.90
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       966
           1       0.89      0.89      0.89       841

    accuracy                           0.90      1807
   macro avg       0.90      0.90      0.90      1807
weighted avg       0.90      0.90      0.90      1807


Test Data Metrics:
Accuracy: 0.83
Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.87      0.84       233
           1       0.85      0.80      0.83       226

    accuracy                           0.83       459
   macro avg       0.84      0.83      0.83       459
weighted avg       0.84 

In [67]:
import gc
gc.collect()

75

task 38: We then test using both PLM and biology features, the results show that using both PLM and biology features shows improved results compared with using only PLM or multi-scale biology features. 

In [70]:
columns_to_keep = [col for col in training_dataset_df.columns if col not in always_exclude_columns]
training_dataset_df[columns_to_keep]

,label,augmentated from uniprot ID,deephase_phys_multi,deephase_w2v_multi,deephase_score,iupred_score,onehot_-7_A,onehot_-7_C,onehot_-7_D,onehot_-7_E,...,ptmmamba_residue_embedding_759,ptmmamba_residue_embedding_760,ptmmamba_residue_embedding_761,ptmmamba_residue_embedding_762,ptmmamba_residue_embedding_763,ptmmamba_residue_embedding_764,ptmmamba_residue_embedding_765,ptmmamba_residue_embedding_766,ptmmamba_residue_embedding_767,phospho_score
0,1,P16333,0.203,0.160,0.182,0.541878,0.0,0.0,0.0,0.0,...,-0.930157,-0.784488,1.378070,1.125187,-0.962103,0.749272,-0.813084,0.488083,1.446624,0.526
1,1,P16333,0.185,0.173,0.179,0.509769,0.0,0.0,0.0,0.0,...,-0.833150,-0.543848,1.270835,1.192153,-0.817320,0.709415,-1.378276,0.626975,2.133350,0.579
2,1,P16333,0.096,0.129,0.113,0.465241,0.0,0.0,0.0,0.0,...,-1.474621,-0.104392,1.062815,0.442375,-0.308503,-0.050680,-1.171706,1.861005,0.189925,0.571
3,1,P16333,0.108,0.352,0.230,0.566480,0.0,0.0,0.0,0.0,...,-0.708835,-0.368524,0.630111,0.287115,0.439652,0.259266,-1.870489,2.179887,0.830376,0.379
4,1,Q5VWQ8,0.282,0.742,0.512,0.837511,0.0,0.0,0.0,0.0,...,-0.940106,-0.094523,0.474609,-0.267281,-0.266375,0.243640,-0.773728,1.224039,-0.136662,0.374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2261,0,O15417,0.329,0.774,0.552,0.798249,0.0,0.0,0.0,0.0,...,-1.177676,-0.877304,0.674698,-0.195342,0.283925,0.078973,0.930381,0.637896,-0.366109,0.080
2262,0,Q9BYW2,0.283,0.776,0.530,0.831250,0.0,0.0,0.0,0.0,...,-1.549038,-0.295345,-0.262526,-0.634858,-0.357382,0.593850,-0.580129,0.019379,-0.784574,0.274
2263,0,Q8N3K9,0.203,0.726,0.464,0.566480,0.0,0.0,0.0,0.0,...,-0.909634,-0.845351,-0.325931,-0.011938,-0.855250,-0.535025,0.514734,0.020328,-0.189157,0.063
2264,0,Q6ZRS2,0.411,0.762,0.587,0.930790,0.0,0.0,0.0,0.0,...,-1.190711,0.183619,1.008728,-0.325716,-1.453191,-0.757268,-0.204297,0.862000,-1.027532,0.155


In [71]:
print(f"Modeling using both PLM and biology features")
model = train(training_dataset_df[columns_to_keep], RandomForestClassifier(max_depth=7))

Modeling using both PLM and biology features
Label counts in y_train:
label
0    966
1    841
Name: count, dtype: int64

Label counts in y_test:
label
0    233
1    226
Name: count, dtype: int64
Model RandomForestClassifier(max_depth=7) Performance Metrics:

Training Data Metrics:
Accuracy: 0.97
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.97      0.97       966
           1       0.96      0.98      0.97       841

    accuracy                           0.97      1807
   macro avg       0.97      0.97      0.97      1807
weighted avg       0.97      0.97      0.97      1807


Test Data Metrics:
Accuracy: 0.79
Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.93      0.82       233
           1       0.90      0.65      0.76       226

    accuracy                           0.79       459
   macro avg       0.82      0.79      0.79       459
weighted avg       0.82

In [72]:
import gc
gc.collect()

75

task 39: feature selection using mRMR. We mainly test how many features are optimized for our model by using different number of features to fit the model and compare the model performance.

In [73]:
# !pip install mrmr_selection

In [82]:
from mrmr import mrmr_classif
from sklearn.metrics import accuracy_score, classification_report, matthews_corrcoef
def mrmr_search_feature_number(df, classifier):
    y = df["label"]
    target = "label"
    features = list(df.columns)
    features.remove(target)
    x = df[features]

    # Split data into train and test sets
    X_train, X_test, y_train, y_test = balanced_group_train_test_split(
        x,
        y,
        group_col='augmentated from uniprot ID',
        test_size=0.2,
        random_state=42
    )

    feature_number_lst = [5,10,20,25,30,40,50,100,200]
    for feature_number in feature_number_lst:
        # using mrmr to get the selected features order
        selected_features = mrmr_classif(X=X_train, y=y_train, K=feature_number)
        print(f"selected feature number is {len(selected_features)}")
        print(selected_features)

        X_train_temp = (X_train).copy()[selected_features]
        X_test_temp = X_test.copy()[selected_features]

        trained_model = make_pipeline(classifier)
        trained_model.fit(X_train_temp, y_train)

        # Make predictions
        y_train_pred = trained_model.predict(X_train_temp)
        y_test_pred = trained_model.predict(X_test_temp)

        # Calculate model metrics
        train_accuracy = accuracy_score(y_train, y_train_pred)
        test_accuracy = accuracy_score(y_test, y_test_pred)
        
        train_mcc = matthews_corrcoef(y_train, y_train_pred)
        test_mcc = matthews_corrcoef(y_test, y_test_pred)
        
        train_report = classification_report(y_train, y_train_pred, zero_division=1)
        test_report = classification_report(y_test, y_test_pred, zero_division=1)
        
        # Print metrics in a readable format
        print("Model Performance Metrics:\n")
        
        print("Training Data Metrics:")
        print(f"Accuracy: {train_accuracy:.2f}")
        print(f"MCC: {train_mcc:.2f}")
        print("Classification Report:")
        print(train_report)
        
        print("\nTest Data Metrics:")
        print(f"Accuracy: {test_accuracy:.2f}")
        print(f"MCC: {test_mcc:.2f}")
        print("Classification Report:")
        print(test_report)
        
        print(f"\nSuccessfully trained the model with {len(selected_features)} feature{'s' if len(selected_features) > 1 else ''}.")

In [83]:
# first mrmr selection on the biological features

In [84]:
print("biological features mrmr selection")
mrmr_search_feature_number(biology_training_dataset_df, RandomForestClassifier(max_depth=7))

biological features mrmr selection


100%|██████████| 5/5 [00:00<00:00, 24.27it/s]


selected feature number is 5
['onehot_-3_R', 'onehot_-1_K', 'onehot_2_P', 'phospho_score', 'onehot_-2_S']
Model Performance Metrics:

Training Data Metrics:
Accuracy: 0.88
MCC: 0.76
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.87      0.88       966
           1       0.85      0.89      0.87       841

    accuracy                           0.88      1807
   macro avg       0.88      0.88      0.88      1807
weighted avg       0.88      0.88      0.88      1807


Test Data Metrics:
Accuracy: 0.83
MCC: 0.65
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.85      0.83       233
           1       0.84      0.80      0.82       226

    accuracy                           0.83       459
   macro avg       0.83      0.83      0.83       459
weighted avg       0.83      0.83      0.83       459


Successfully trained the model with 5 features.


100%|██████████| 10/10 [00:00<00:00, 29.83it/s]


selected feature number is 10
['onehot_-3_R', 'onehot_-1_K', 'onehot_2_P', 'phospho_score', 'onehot_-2_S', 'onehot_1_L', 'onehot_0_t', 'onehot_-3_E', 'fR', 'onehot_1_P']
Model Performance Metrics:

Training Data Metrics:
Accuracy: 0.89
MCC: 0.78
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.88      0.90       966
           1       0.87      0.90      0.88       841

    accuracy                           0.89      1807
   macro avg       0.89      0.89      0.89      1807
weighted avg       0.89      0.89      0.89      1807


Test Data Metrics:
Accuracy: 0.84
MCC: 0.68
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       233
           1       0.85      0.82      0.83       226

    accuracy                           0.84       459
   macro avg       0.84      0.84      0.84       459
weighted avg       0.84      0.84      0.84       459


Successf

...

selected feature number is 50
['onehot_-3_R', 'onehot_-1_K', 'onehot_2_P', 'phospho_score', 'onehot_-2_S', 'onehot_1_L', 'onehot_0_t', 'onehot_-3_E', 'fR', 'onehot_1_P', 'onehot_4_L', 'onehot_-3_S', 'onehot_-4_R', 'iupred_score', 'onehot_-2_G', 'onehot_-3_V', 'onehot_2_K', 'onehot_-2_R', 'onehot_-1_H', 'onehot_0_s', 'onehot_1_R', 'onehot_-3_G', 'SHD', 'onehot_-3_A', 'onehot_-3_D', 'onehot_-5_R', 'onehot_-3_N', 'onehot_-3_P', 'onehot_3_D', 'onehot_-3_L', 'FCR', 'onehot_-3_Q', 'onehot_2_A', 'onehot_4_Q', 'deephase_w2v_multi', 'NCPR', 'onehot_-2_D', 'mean_lambda', 'onehot_1_E', 'onehot_-2_N', 'onehot_-5_N', 'onehot_-1_G', 'onehot_2_I', 'onehot_1_G', 'onehot_2_F', 'onehot_-1_D', 'SconfSVR/N (kB)', 'onehot_-2_L', 'onehot_7_G', 'onehot_5_H']
Model Performance Metrics:

Training Data Metrics:
Accuracy: 0.90
MCC: 0.80
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.90      0.91       966
           1       0.89      0.90      0.89   

100%|██████████| 100/100 [00:03<00:00, 28.09it/s]


selected feature number is 100
['onehot_-3_R', 'onehot_-1_K', 'onehot_2_P', 'phospho_score', 'onehot_-2_S', 'onehot_1_L', 'onehot_0_t', 'onehot_-3_E', 'fR', 'onehot_1_P', 'onehot_4_L', 'onehot_-3_S', 'onehot_-4_R', 'iupred_score', 'onehot_-2_G', 'onehot_-3_V', 'onehot_2_K', 'onehot_-2_R', 'onehot_-1_H', 'onehot_0_s', 'onehot_1_R', 'onehot_-3_G', 'SHD', 'onehot_-3_A', 'onehot_-3_D', 'onehot_-5_R', 'onehot_-3_N', 'onehot_-3_P', 'onehot_3_D', 'onehot_-3_L', 'FCR', 'onehot_-3_Q', 'onehot_2_A', 'onehot_4_Q', 'deephase_w2v_multi', 'NCPR', 'onehot_-2_D', 'mean_lambda', 'onehot_1_E', 'onehot_-2_N', 'onehot_-5_N', 'onehot_-1_G', 'onehot_2_I', 'onehot_1_G', 'onehot_2_F', 'onehot_-1_D', 'SconfSVR/N (kB)', 'onehot_-2_L', 'onehot_7_G', 'onehot_5_H', 'onehot_-1_N', 'onehot_1_A', 'onehot_-2_E', 'nuSVR', 'onehot_-4_K', 'onehot_-5_T', 'deephase_score', 'onehot_2_L', 'onehot_-2_F', 'onehot_-3_W', 'onehot_-6_G', 'onehot_-2_V', 'onehot_7_W', 'onehot_-1_E', 'onehot_3_R', 'onehot_-2_P', 'onehot_-4_G', 'oneh

100%|██████████| 200/200 [00:06<00:00, 31.81it/s]


selected feature number is 200
['onehot_-3_R', 'onehot_-1_K', 'onehot_2_P', 'phospho_score', 'onehot_-2_S', 'onehot_1_L', 'onehot_0_t', 'onehot_-3_E', 'fR', 'onehot_1_P', 'onehot_4_L', 'onehot_-3_S', 'onehot_-4_R', 'iupred_score', 'onehot_-2_G', 'onehot_-3_V', 'onehot_2_K', 'onehot_-2_R', 'onehot_-1_H', 'onehot_0_s', 'onehot_1_R', 'onehot_-3_G', 'SHD', 'onehot_-3_A', 'onehot_-3_D', 'onehot_-5_R', 'onehot_-3_N', 'onehot_-3_P', 'onehot_3_D', 'onehot_-3_L', 'FCR', 'onehot_-3_Q', 'onehot_2_A', 'onehot_4_Q', 'deephase_w2v_multi', 'NCPR', 'onehot_-2_D', 'mean_lambda', 'onehot_1_E', 'onehot_-2_N', 'onehot_-5_N', 'onehot_-1_G', 'onehot_2_I', 'onehot_1_G', 'onehot_2_F', 'onehot_-1_D', 'SconfSVR/N (kB)', 'onehot_-2_L', 'onehot_7_G', 'onehot_5_H', 'onehot_-1_N', 'onehot_1_A', 'onehot_-2_E', 'nuSVR', 'onehot_-4_K', 'onehot_-5_T', 'deephase_score', 'onehot_2_L', 'onehot_-2_F', 'onehot_-3_W', 'onehot_-6_G', 'onehot_-2_V', 'onehot_7_W', 'onehot_-1_E', 'onehot_3_R', 'onehot_-2_P', 'onehot_-4_G', 'oneh

In [85]:
# then mrmr selection on all the features

In [86]:
training_dataset_df = pd.read_pickle("final_dataset_with_all_new.pkl")
training_dataset_df = training_dataset_df[training_dataset_df["usage"] == "training"]
columns_to_keep = [col for col in training_dataset_df.columns if col not in always_exclude_columns]
training_dataset_df[columns_to_keep]

,label,augmentated from uniprot ID,deephase_phys_multi,deephase_w2v_multi,deephase_score,iupred_score,onehot_-7_A,onehot_-7_C,onehot_-7_D,onehot_-7_E,...,ptmmamba_residue_embedding_759,ptmmamba_residue_embedding_760,ptmmamba_residue_embedding_761,ptmmamba_residue_embedding_762,ptmmamba_residue_embedding_763,ptmmamba_residue_embedding_764,ptmmamba_residue_embedding_765,ptmmamba_residue_embedding_766,ptmmamba_residue_embedding_767,phospho_score
0,1,P16333,0.203,0.160,0.182,0.541878,0.0,0.0,0.0,0.0,...,-0.930157,-0.784488,1.378070,1.125187,-0.962103,0.749272,-0.813084,0.488083,1.446624,0.526
1,1,P16333,0.185,0.173,0.179,0.509769,0.0,0.0,0.0,0.0,...,-0.833150,-0.543848,1.270835,1.192153,-0.817320,0.709415,-1.378276,0.626975,2.133350,0.579
2,1,P16333,0.096,0.129,0.113,0.465241,0.0,0.0,0.0,0.0,...,-1.474621,-0.104392,1.062815,0.442375,-0.308503,-0.050680,-1.171706,1.861005,0.189925,0.571
3,1,P16333,0.108,0.352,0.230,0.566480,0.0,0.0,0.0,0.0,...,-0.708835,-0.368524,0.630111,0.287115,0.439652,0.259266,-1.870489,2.179887,0.830376,0.379
4,1,Q5VWQ8,0.282,0.742,0.512,0.837511,0.0,0.0,0.0,0.0,...,-0.940106,-0.094523,0.474609,-0.267281,-0.266375,0.243640,-0.773728,1.224039,-0.136662,0.374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2261,0,O15417,0.329,0.774,0.552,0.798249,0.0,0.0,0.0,0.0,...,-1.177676,-0.877304,0.674698,-0.195342,0.283925,0.078973,0.930381,0.637896,-0.366109,0.080
2262,0,Q9BYW2,0.283,0.776,0.530,0.831250,0.0,0.0,0.0,0.0,...,-1.549038,-0.295345,-0.262526,-0.634858,-0.357382,0.593850,-0.580129,0.019379,-0.784574,0.274
2263,0,Q8N3K9,0.203,0.726,0.464,0.566480,0.0,0.0,0.0,0.0,...,-0.909634,-0.845351,-0.325931,-0.011938,-0.855250,-0.535025,0.514734,0.020328,-0.189157,0.063
2264,0,Q6ZRS2,0.411,0.762,0.587,0.930790,0.0,0.0,0.0,0.0,...,-1.190711,0.183619,1.008728,-0.325716,-1.453191,-0.757268,-0.204297,0.862000,-1.027532,0.155


In [87]:
mrmr_search_feature_number(training_dataset_df[columns_to_keep], RandomForestClassifier(max_depth=7))

100%|██████████| 5/5 [00:01<00:00,  3.78it/s]


selected feature number is 5
['onehot_-3_R', 'ptmmamba_residue_embedding_79', 'ptmmamba_residue_embedding_408', 'esm_residue_embedding_81', 'onehot_2_P']
Model Performance Metrics:

Training Data Metrics:
Accuracy: 0.89
MCC: 0.78
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.87      0.89       966
           1       0.86      0.90      0.88       841

    accuracy                           0.89      1807
   macro avg       0.89      0.89      0.89      1807
weighted avg       0.89      0.89      0.89      1807


Test Data Metrics:
Accuracy: 0.83
MCC: 0.66
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.84      0.84       233
           1       0.83      0.82      0.83       226

    accuracy                           0.83       459
   macro avg       0.83      0.83      0.83       459
weighted avg       0.83      0.83      0.83       459


Successfully trained the

100%|██████████| 10/10 [00:02<00:00,  3.34it/s]


selected feature number is 10
['onehot_-3_R', 'ptmmamba_residue_embedding_79', 'ptmmamba_residue_embedding_408', 'esm_residue_embedding_81', 'onehot_2_P', 'esm_residue_embedding_1041', 'esm_residue_embedding_715', 'esm_residue_embedding_1171', 'phospho_score', 'esm_residue_embedding_491']
Model Performance Metrics:

Training Data Metrics:
Accuracy: 0.92
MCC: 0.84
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.91      0.92       966
           1       0.90      0.93      0.91       841

    accuracy                           0.92      1807
   macro avg       0.92      0.92      0.92      1807
weighted avg       0.92      0.92      0.92      1807


Test Data Metrics:
Accuracy: 0.86
MCC: 0.71
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       233
           1       0.86      0.85      0.85       226

    accuracy                           0.86       45

100%|██████████| 20/20 [00:06<00:00,  3.14it/s]


...

selected feature number is 50
['onehot_-3_R', 'ptmmamba_residue_embedding_79', 'ptmmamba_residue_embedding_408', 'esm_residue_embedding_81', 'onehot_2_P', 'esm_residue_embedding_1041', 'esm_residue_embedding_715', 'esm_residue_embedding_1171', 'phospho_score', 'esm_residue_embedding_491', 'esm_residue_embedding_1259', 'esm_residue_embedding_394', 'esm_residue_embedding_1215', 'esm_residue_embedding_53', 'esm_residue_embedding_546', 'esm_residue_embedding_356', 'esm_residue_embedding_197', 'esm_residue_embedding_327', 'esm_residue_embedding_385', 'esm_residue_embedding_453', 'esm_protein_embedding_68', 'esm_residue_embedding_1242', 'esm_residue_embedding_482', 'esm_residue_embedding_844', 'esm_residue_embedding_1074', 'esm_residue_embedding_1042', 'esm_residue_embedding_333', 'esm_residue_embedding_692', 'esm_residue_embedding_313', 'esm_residue_embedding_960', 'esm_residue_embedding_2', 'esm_residue_embedding_291', 'esm_residue_embedding_650', 'esm_residue_embedding_1110', 'esm_residue

100%|██████████| 100/100 [00:34<00:00,  2.94it/s]


selected feature number is 100
['onehot_-3_R', 'ptmmamba_residue_embedding_79', 'ptmmamba_residue_embedding_408', 'esm_residue_embedding_81', 'onehot_2_P', 'esm_residue_embedding_1041', 'esm_residue_embedding_715', 'esm_residue_embedding_1171', 'phospho_score', 'esm_residue_embedding_491', 'esm_residue_embedding_1259', 'esm_residue_embedding_394', 'esm_residue_embedding_1215', 'esm_residue_embedding_53', 'esm_residue_embedding_546', 'esm_residue_embedding_356', 'esm_residue_embedding_197', 'esm_residue_embedding_327', 'esm_residue_embedding_385', 'esm_residue_embedding_453', 'esm_protein_embedding_68', 'esm_residue_embedding_1242', 'esm_residue_embedding_482', 'esm_residue_embedding_844', 'esm_residue_embedding_1074', 'esm_residue_embedding_1042', 'esm_residue_embedding_333', 'esm_residue_embedding_692', 'esm_residue_embedding_313', 'esm_residue_embedding_960', 'esm_residue_embedding_2', 'esm_residue_embedding_291', 'esm_residue_embedding_650', 'esm_residue_embedding_1110', 'esm_residu

100%|██████████| 200/200 [01:07<00:00,  2.96it/s]


selected feature number is 200
['onehot_-3_R', 'ptmmamba_residue_embedding_79', 'ptmmamba_residue_embedding_408', 'esm_residue_embedding_81', 'onehot_2_P', 'esm_residue_embedding_1041', 'esm_residue_embedding_715', 'esm_residue_embedding_1171', 'phospho_score', 'esm_residue_embedding_491', 'esm_residue_embedding_1259', 'esm_residue_embedding_394', 'esm_residue_embedding_1215', 'esm_residue_embedding_53', 'esm_residue_embedding_546', 'esm_residue_embedding_356', 'esm_residue_embedding_197', 'esm_residue_embedding_327', 'esm_residue_embedding_385', 'esm_residue_embedding_453', 'esm_protein_embedding_68', 'esm_residue_embedding_1242', 'esm_residue_embedding_482', 'esm_residue_embedding_844', 'esm_residue_embedding_1074', 'esm_residue_embedding_1042', 'esm_residue_embedding_333', 'esm_residue_embedding_692', 'esm_residue_embedding_313', 'esm_residue_embedding_960', 'esm_residue_embedding_2', 'esm_residue_embedding_291', 'esm_residue_embedding_650', 'esm_residue_embedding_1110', 'esm_residu

task 40: We also test feature selection using another method Boruta, it shows that XX feature give us the best results.

In [ ]:
# initial parameters
def boruta_feature_selection(df, classifier)
    y = df["label"]
    target = "label"
    features = list(df.columns)
    features.remove(target)
    x = df[features]

    # Split data into train and test sets
    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )


    alpha = 0.01
    perc = 100
    max_iter = 80

    features = X_train.columns
    tree_model = classifier
    X_train_temp = X_train.copy()

    # Boruta search
    feat_selector = BorutaPy(tree_model, 
                             n_estimators='auto', 
                             verbose=2, 
                             random_state=0, 
                             perc = perc, 
                             max_iter = max_iter, 
                             alpha = alpha)

    feat_selector.fit(X_train_temp, y_train)

    # get the feature that will be deleted
    rank_mask = feat_selector.ranking_
    deleted_features = [X_train_temp.columns[i] for i, num in enumerate(rank_mask) if (num != 2 and num !=1)]

    # Convert the result set to a string
    result_str = ', '.join(map(str, set(features)-set(deleted_features)))

    # Open a file in write mode and save the result
    with open('20240517 boruta selected features.txt', 'w') as file:
        file.write(result_str)

    # if I want to control the number of feature to selected
    remaining_feature_number = len(X_train.columns) - len(deleted_features)
    print("remaining feature number: ", remaining_feature_number)
    print("remaining feature", set(features)-set(deleted_features))
    return set(features)-set(deleted_features)

In [ ]:
selected_features = boruta_feature_selection(training_dataset_df)

task 41: feature importance and its contribution were
further analyzed to find which feature was more valuable for
the model performance after feature selection 

In [ ]:
# Prefixes for PLM features
plm_prefixes = (
    'esm_protein_embedding_position',
    'esm_residue_embedding_position',
    'ptmmamba_residue_embedding_position',
    'ptmmamba_protein_embedding_position'
)

# Identify PLM and Biological features
plm_features = [feature for feature in selected_features if feature.startswith(plm_prefixes)]
bio_features = [feature for feature in selected_features if not feature.startswith(plm_prefixes)]

# Counts
plm_count = len(plm_features)
bio_count = len(bio_features)

# Output
print(f"PLM features ({plm_count}): {plm_features}")
print(f"Biological features ({bio_count}): {bio_features}")

print(f"PLM features: {plm_count}")
print(f"Biological features: {bio_count}")

### 4.4	Prediction performance with different classifiers

task 42: To test the validity of the optimal feature set in different classifiers, three common classifiers were used to predict 14-3-3 sites: random forest (RF), XGBoost (XGB), SVM and ANN are used. 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

classifer_lst = [RandomForestClassifier(), XGBClassifier(), SVC(), MLPClassifier()]

for classifer in classifer_lst:
    train(df, classifier)

## 4.5	Model architecture and optimization 

task 43: fine-tuned certain hyperparameters (Table X) of the model

In [ ]:
y = df["label"]
target = "label"
features = list(df.columns)
features.remove(target)
x = df[features]

# Split data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

steps = [('classifier', XGBClassifier())]
pipe = Pipeline(steps)

In [ ]:
parameters = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [2,3,4],
    'classifier__gamma': [0, 0.5, 1],
    'classifier__reg_alpha': [0, 0.5, 1],
    'classifier__reg_lambda': [0.5, 1, 5],
    'classifier__base_score': [0.2, 0.5, 1]
}

In [ ]:
scorer = make_scorer(average_precision_score)
model_gsv = GridSearchCV(pipe, parameters, cv = 5, scoring = scorer, n_jobs=-1, verbose=0)
model_gsv = model_gsv.fit(X_train, y_train)
model_gsv.best_params_
best_xgboost_model = model_gsv.best_estimator_

In [ ]:
model_gsv.best_params_

In [ ]:
# Evaluate the best model on the test set
y_train_proba = best_xgboost_model.predict_proba(X_train)[:, 1]
y_train_pred = best_xgboost_model.predict(X_train)
    
model_classification_report = classification_report(y_train, y_train_pred)
    
y_valid_proba = best_xgboost_model.predict_proba(X_valid)[:, 1]
y_valid_pred = best_xgboost_model.predict(X_valid)    
    
model_classification_report_valid = classification_report(y_valid, y_valid_pred)

print("XGBoost Model")
print("Training performance:")
print(f"average_precision_score: {average_precision_score(y_train, y_train_proba):.3e}")
print(f"model_classification_report of traning data:")
print(model_classification_report)
print("-" * 20)
print("Testing performance:")
print(f"average_precision_score: {average_precision_score(y_valid, y_valid_proba):.3e}")
print(f"model_classification_report of valida data:")
print(model_classification_report_valid)
print("-" * 20)

## 4.6 Feature Importance analysis

task 44: we conducted a thorough examination of feature importance using SHAP (SHapley Additive exPlanations) values

In [ ]:
best_model = best_xgboost_model

In [ ]:
select_feature_instance = best_model.named_steps['Drop']

# Accessing the mi_select_cols attribute
feature_names = select_feature_instance.select_cols

feature_importances = best_model.named_steps['classifier'].feature_importances_

# Print or use mi_select_cols as needed
print("Selected features: ", feature_names)

feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
feature_importance_df[:20]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Sorting the DataFrame based on the 'Value' column in descending order
df_sorted = feature_importance_df.sort_values(by='Importance', ascending=True)

# Plotting the data
plt.figure(figsize=(10, 6))  # Set the figure size
plt.barh(df_sorted.iloc[:, 0], df_sorted.iloc[:, 1], color='skyblue')  # Create a horizontal bar plot

# Adding title and labels
plt.title('Feature vs Importance')
plt.ylabel(df_sorted.columns[0])
plt.xlabel(df_sorted.columns[1])

# Display the plot
plt.show()

In [ ]:
# !pip install shap
import shap

random_forest_model = best_model.named_steps['classifier']
# print(type(random_forest_model))
# Create a SHAP explainer

# Create a SHAP explainer
explainer = shap.TreeExplainer(random_forest_model)

# Calculate SHAP values
shap_values = explainer(df_drop[feature_names])


# Visualize feature importance with feature values
shap.plots.beeswarm(shap_values, show=True)

task 45: ablation experiments where we systematically removed features to observe changes in model performance.

## 4.7	Performance evaluation and comparison with existing methods, Validation on independent test sets

task 46: Independent datset test with 14-3-3 pred and 14-3-3 site-finder, compare with our results.